In [ ]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import Row

In [ ]:
# Robustly stop any existing SparkContext before starting a new one
try:
  if 'SpContext' in locals() or 'SpContext' in globals():
    SpContext.stop()
  SparkContext.getOrCreate().stop()
except:
    pass

In [ ]:
print("=== RDD-based decision tree learning for breast cancer ===")

# Create a SparkContext
conf = SparkConf().setMaster("local").setAppName("DTBreastCancer")
SpContext = SparkContext(conf = conf)

=== RDD-based decision tree learning for breast cancer ===


In [ ]:
SpSession = SparkSession.builder \
    .appName("DTBreastCancer") \
    .config("spark.sql.warehouse.dir", "/content") \
    .getOrCreate()

In [ ]:
!wget -O 8-breast-cancer.txt https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-breast-cancer.txt

--2026-04-07 00:39:52--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/8-breast-cancer.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 82965 (81K) [text/plain]
Saving to: ‘8-breast-cancer.txt’

8-breast-cancer.txt 100%[===================>]  81.02K  --.-KB/s    in 0.02s   

2026-04-07 00:39:52 (5.02 MB/s) - ‘8-breast-cancer.txt’ saved [82965/82965]



In [ ]:
cancerData = SpContext.textFile('8-breast-cancer.txt')
cancerData.cache()

8-breast-cancer.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0

In [ ]:
# Filter out empty lines to avoid ValueError during conversion
dataLines = cancerData.filter(lambda x: len(x) > 0)
print("Total count of data: "+ str(dataLines.count()))

print("=== Reading data ===")
from pyspark.sql import Row
#Create a Data Frame from the data
# Use re.split to split by one or more whitespace characters
parts = dataLines.map(lambda l: re.split(r'\s+', l.strip()))

# Add a check to ensure the line has enough parts (1 label + 10 features = 11 parts)
cancerMap = parts.filter(lambda p: len(p) == 11).map(lambda p: Row(OUTCOME=float(p[0]),\
                                RADIUS=float(p[1].split(':')[1]), \
                                TEXTURE=float(p[2].split(':')[1]), \
                                PERIMETER=float(p[3].split(':')[1]), \
                                AREA=float(p[4].split(':')[1]), \
                                SMOOTHNESS=float(p[5].split(':')[1]), \
                                COMPACTNESS=float(p[6].split(':')[1]), \
                                CONCAVITY=float(p[7].split(':')[1]), \
                                CONCAVE_POINTS=float(p[8].split(':')[1]), \
                                SYMMETRY=float(p[9].split(':')[1]), \
                                FRACTAL_DIMENSION=float(p[10].split(':')[1])))

# Infer the schema, and register the DataFrame as a table.
cancerDf = SpSession.createDataFrame(cancerMap)
cancerDf.cache()

# No need for StringIndexer as OUTCOME is already numeric (0 or 1)
cancerNormDf = cancerDf
cancerNormDf.cache()

cancerNormDf.select("OUTCOME").distinct().collect()
cancerNormDf.describe().show(10)

Total count of data: 683
=== Reading data ===
+-------+-------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+
|summary|            OUTCOME|            RADIUS|           TEXTURE|         PERIMETER|              AREA|       SMOOTHNESS|       COMPACTNESS|         CONCAVITY|   CONCAVE_POINTS|          SYMMETRY| FRACTAL_DIMENSION|
+-------+-------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+
|  count|                683|               683|               683|               683|               683|              683|               683|               683|              683|               683|               683|
|   mean|0.34992679355783307|1076720.2269399706|  4.44216691068814| 3.150805270863

In [ ]:
print("=== Data analysis ===")

#Find correlation between predictors and target
for i in cancerNormDf.columns:
    if not( isinstance(cancerNormDf.select(i).take(1)[0][0], str)) and i != "OUTCOME":
        print( "Correlation to OUTCOME for ", i, \
                    cancerNormDf.stat.corr('OUTCOME',i))

=== Data analysis ===
Correlation to OUTCOME for  RADIUS -0.08470102742395866
Correlation to OUTCOME for  TEXTURE 0.71478992632216
Correlation to OUTCOME for  PERIMETER 0.8208014428258774
Correlation to OUTCOME for  AREA 0.8218909476888678
Correlation to OUTCOME for  SMOOTHNESS 0.7062941354660842
Correlation to OUTCOME for  COMPACTNESS 0.6909581590873206
Correlation to OUTCOME for  CONCAVITY 0.8226958729964622
Correlation to OUTCOME for  CONCAVE_POINTS 0.7582275545334304
Correlation to OUTCOME for  SYMMETRY 0.7186771878756356
Correlation to OUTCOME for  FRACTAL_DIMENSION 0.4234479212952123


In [ ]:
print("=== Prepare data for ML ===")
#Transform to a Data Frame for input to Machine Learing
#Drop columns that are not required (low correlation)

from pyspark.ml.linalg import Vectors
def transformToLabeledPoint(row) :
    lp = ( row["OUTCOME"], \
                Vectors.dense([row["RADIUS"],\
                        row["TEXTURE"], \
                        row["PERIMETER"], \
                        row["AREA"], \
                        row["SMOOTHNESS"], \
                        row["COMPACTNESS"], \
                        row["CONCAVITY"], \
                        row["CONCAVE_POINTS"], \
                        row["SYMMETRY"], \
                        row["FRACTAL_DIMENSION"]]))
    return lp

cancerLp = cancerNormDf.rdd.map(transformToLabeledPoint)
cancerLpDf = SpSession.createDataFrame(cancerLp,["outcome_label", "features"])
cancerLpDf.select("outcome_label","features").orderBy("features").show()
cancerLpDf.cache()

=== Prepare data for ML ===
+-------------+--------------------+
|outcome_label|            features|
+-------------+--------------------+
|          1.0|[63375.0,9.0,1.0,...|
|          1.0|[76389.0,10.0,4.0...|
|          1.0|[95719.0,6.0,10.0...|
|          0.0|[128059.0,1.0,1.0...|
|          1.0|[142932.0,7.0,6.0...|
|          1.0|[144888.0,8.0,10....|
|          1.0|[145447.0,8.0,4.0...|
|          1.0|[160296.0,5.0,8.0...|
|          0.0|[167528.0,4.0,1.0...|
|          0.0|[183913.0,1.0,2.0...|
|          0.0|[183936.0,3.0,1.0...|
|          1.0|[188336.0,5.0,3.0...|
|          1.0|[191250.0,10.0,4....|
|          0.0|[242970.0,5.0,7.0...|
|          1.0|[255644.0,10.0,5....|
|          1.0|[263538.0,5.0,10....|
|          1.0|[274137.0,8.0,8.0...|
|          1.0|[303213.0,10.0,4....|
|          1.0|[314428.0,7.0,9.0...|
|          1.0|[320675.0,3.0,3.0...|
+-------------+--------------------+
only showing top 20 rows


DataFrame[outcome_label: double, features: vector]

In [ ]:
print("=== Building model ===")

(trainingData, testData) = cancerLpDf.randomSplit([0.8, 0.2])
trainingData.count()
testData.count()
testData.collect()

#import the decision tree algorithm class and its evaluator
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Create the model
dtClassifer = DecisionTreeClassifier(impurity='gini', maxDepth = 8, maxBins = 100, labelCol="outcome_label",featuresCol="features")
dtModel = dtClassifer.fit(trainingData)

=== Building model ===


In [ ]:
print("=== Learned model for breast cancer ===")

nodeCount = dtModel.numNodes
depth = dtModel.depth
treeRules = dtModel

print("nodeCount: "+str(nodeCount)) #show how many nodes were learned
print("depth: "+str(depth)) #show the depth of the tree
print(str(treeRules.toDebugString)) #show the learned tree as rules

=== Learned model for breast cancer ===
nodeCount: 53
depth: 8
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_fa2ae1602466, depth=8, numNodes=53, numClasses=2, numFeatures=10
  If (feature 2 <= 2.5)
   If (feature 6 <= 5.5)
    If (feature 8 <= 8.5)
     If (feature 1 <= 6.5)
      If (feature 6 <= 4.5)
       Predict: 0.0
      Else (feature 6 > 4.5)
       If (feature 5 <= 1.5)
        Predict: 1.0
       Else (feature 5 > 1.5)
        Predict: 0.0
     Else (feature 1 > 6.5)
      If (feature 3 <= 2.5)
       Predict: 0.0
      Else (feature 3 > 2.5)
       Predict: 1.0
    Else (feature 8 > 8.5)
     Predict: 1.0
   Else (feature 6 > 5.5)
    If (feature 1 <= 1.5)
     Predict: 0.0
    Else (feature 1 > 1.5)
     Predict: 1.0
  Else (feature 2 > 2.5)
   If (feature 6 <= 1.5)
    If (feature 2 <= 3.5)
     Predict: 0.0
    Else (feature 2 > 3.5)
     If (feature 5 <= 3.5)
      Predict: 0.0
     Else (feature 5 > 3.5)
      If (feature 0 <= 1296298.5)
       Predict: 1.

In [ ]:
print("=== Evaluation reults ===")

#Predict on the test data
predictions = dtModel.transform(testData)
predictions.select("prediction","outcome_label").collect()

#Evaluate precision
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction", \
                    labelCol="outcome_label",metricName="weightedPrecision")
precision = evaluator.evaluate(predictions)
print("The precision of the model is: "+str(precision))


#Draw a confusion matrix
predictions.groupBy("outcome_label","prediction").count().show()

SpSession.stop()

=== Evaluation reults ===
The precision of the model is: 0.9862058951648033
+-------------+----------+-----+
|outcome_label|prediction|count|
+-------------+----------+-----+
|          1.0|       1.0|   45|
|          1.0|       0.0|    2|
|          0.0|       0.0|   95|
+-------------+----------+-----+

